# Mini Model V2 - Correct Dataset Structure
**Based on actual competition data:**
- Training: Instrument stems (drums, vocals, bass, others)
- Test: Noisy mashups (mixed stems + ESC-50 noise)

**Strategy:**
1. Mix stems to simulate mashups during training
2. Add ESC-50 noise for robustness
3. Use research-backed settings (15s, 128 mels, SpecAugment)

In [51]:
!pip install -q librosa timm

import os
import glob
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [52]:
# Config
CONFIG = {
    'sr': 22050,
    'duration': 10,           # Reduced from 15 for speed
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'num_classes': 10,
    'batch_size': 24,         # Reduced for GPU memory
    'epochs': 5,
    'lr': 1e-3,
    'train_samples': 1000,    # Start small to validate
    'noise_prob': 0.7,
    'noise_level': (0.05, 0.3),
    'mixup_alpha': 0.3,
    'label_smoothing': 0.1,
}

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']  # NOTE: 'other' not 'others'

print(f"Config: {CONFIG['train_samples']} samples, {CONFIG['epochs']} epochs, batch={CONFIG['batch_size']}")

Config: 1000 samples, 5 epochs, batch=24


In [53]:
# Paths - CORRECT for this competition
BASE_PATH = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR = os.path.join(BASE_PATH, 'genres_stems')
NOISE_DIR = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
TEST_DIR = os.path.join(BASE_PATH, 'mashups')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
SAMPLE_SUB = os.path.join(BASE_PATH, 'sample_submission.csv')

print(f"Stems dir: {STEMS_DIR}")
print(f"Noise dir: {NOISE_DIR}")
print(f"Test dir: {TEST_DIR}")

# Verify paths exist
for p in [STEMS_DIR, TEST_DIR]:
    print(f"{p} exists: {os.path.exists(p)}")

Stems dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
Noise dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio
Test dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems exists: True
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups exists: True


In [54]:
# Explore data structure
print("=" * 50)
print("DATA STRUCTURE")
print("=" * 50)

if os.path.exists(STEMS_DIR):
    print(f"\nGenres in stems dir:")
    for genre in sorted(os.listdir(STEMS_DIR)):
        genre_path = os.path.join(STEMS_DIR, genre)
        if os.path.isdir(genre_path):
            songs = os.listdir(genre_path)
            print(f"  {genre}: {len(songs)} songs")
            # Show first song's stems
            if songs:
                first_song = os.path.join(genre_path, songs[0])
                if os.path.isdir(first_song):
                    stems = os.listdir(first_song)
                    print(f"    Sample stems: {stems[:4]}")

if os.path.exists(TEST_DIR):
    test_files = os.listdir(TEST_DIR)
    print(f"\nTest mashups: {len(test_files)} files")
    print(f"  Sample: {test_files[:3]}")

if os.path.exists(NOISE_DIR):
    noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav'))
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

DATA STRUCTURE

Genres in stems dir:
  blues: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  classical: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  country: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  disco: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  hiphop: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  jazz: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  metal: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  pop: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  reggae: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  rock: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

Test mashups: 3020 files
  Sample: ['song2501.wav', 'song0

In [55]:
# Load noise files for augmentation
noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav')) if os.path.exists(NOISE_DIR) else []
print(f"Loaded {len(noise_files)} noise files for augmentation")

def load_noise(sr=22050, duration=15):
    """Load a random noise clip"""
    if not noise_files:
        return np.zeros(sr * duration)
    
    noise_path = random.choice(noise_files)
    try:
        noise, _ = librosa.load(noise_path, sr=sr, duration=duration)
        target_len = sr * duration
        if len(noise) < target_len:
            # Loop noise
            repeats = int(np.ceil(target_len / len(noise)))
            noise = np.tile(noise, repeats)[:target_len]
        else:
            noise = noise[:target_len]
        return noise
    except:
        return np.zeros(sr * duration)

Loaded 2000 noise files for augmentation


In [56]:
# Build training data index
# Structure: genres_stems/genre/song_id/{drums,vocals,bass,others}.wav

train_data = []  # List of (genre, song_path) tuples

for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_dir):
        for song_id in os.listdir(genre_dir):
            song_path = os.path.join(genre_dir, song_id)
            if os.path.isdir(song_path):
                # Verify stems exist
                stems_exist = all(
                    os.path.exists(os.path.join(song_path, f"{stem}.wav"))
                    for stem in STEMS
                )
                if stems_exist:
                    train_data.append((genre, song_path))

print(f"Total songs with all stems: {len(train_data)}")

# Count per genre
genre_counts = {}
for genre, _ in train_data:
    genre_counts[genre] = genre_counts.get(genre, 0) + 1
print(f"Per genre: {genre_counts}")

Total songs with all stems: 1000
Per genre: {'blues': 100, 'classical': 100, 'country': 100, 'disco': 100, 'hiphop': 100, 'jazz': 100, 'metal': 100, 'pop': 100, 'reggae': 100, 'rock': 100}


In [57]:
# Audio loading and preprocessing
def load_and_mix_stems(song_path, sr=22050, duration=15):
    """Load all stems and mix them (simulating mashup creation)"""
    target_len = sr * duration
    mixed = np.zeros(target_len, dtype=np.float32)
    
    for stem in STEMS:
        stem_path = os.path.join(song_path, f"{stem}.wav")
        try:
            audio, _ = librosa.load(stem_path, sr=sr, duration=duration)
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
            mixed += audio
        except:
            pass
    
    # Normalize mixed audio
    if np.max(np.abs(mixed)) > 0:
        mixed = mixed / np.max(np.abs(mixed)) * 0.9
    
    return mixed.astype(np.float32)

def add_noise(audio, noise_level=0.1):
    """Add ESC-50 noise to audio"""
    noise = load_noise(CONFIG['sr'], CONFIG['duration'])
    # Normalize noise
    if np.max(np.abs(noise)) > 0:
        noise = noise / np.max(np.abs(noise))
    # Mix with specified level
    noisy_audio = audio + noise_level * noise
    # Renormalize
    if np.max(np.abs(noisy_audio)) > 0:
        noisy_audio = noisy_audio / np.max(np.abs(noisy_audio)) * 0.9
    return noisy_audio.astype(np.float32)

def preprocess(audio):
    """Basic preprocessing"""
    audio = audio - np.mean(audio)  # DC offset
    rms = np.sqrt(np.mean(audio**2)) + 1e-8
    audio = audio / rms * 0.1  # RMS normalize
    return np.clip(audio, -1, 1).astype(np.float32)

def audio_to_mel(audio, sr=22050):
    """Convert to mel spectrogram"""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=CONFIG['n_mels'],
        n_fft=CONFIG['n_fft'],
        hop_length=CONFIG['hop_length']
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db

In [58]:
# SpecAugment
def spec_augment(mel, time_mask=20, freq_mask=10):
    mel = mel.copy()
    n_mels, n_frames = mel.shape
    
    # Time masking
    if n_frames > time_mask:
        t = np.random.randint(0, time_mask)
        t0 = np.random.randint(0, n_frames - t)
        mel[:, t0:t0+t] = 0
    
    # Frequency masking
    if n_mels > freq_mask:
        f = np.random.randint(0, freq_mask)
        f0 = np.random.randint(0, n_mels - f)
        mel[f0:f0+f, :] = 0
    
    return mel

In [59]:
# Dataset for STEMS
class StemDataset(Dataset):
    def __init__(self, data, augment=False):
        """
        data: list of (genre, song_path) tuples
        """
        self.data = data
        self.augment = augment
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        genre, song_path = self.data[idx]
        label = genre_to_idx[genre]
        
        # Load and mix stems
        audio = load_and_mix_stems(song_path, CONFIG['sr'], CONFIG['duration'])
        
        # Add noise (like test data)
        if self.augment and np.random.random() < CONFIG['noise_prob']:
            noise_level = np.random.uniform(*CONFIG['noise_level'])
            audio = add_noise(audio, noise_level)
        
        # Basic audio augmentation
        if self.augment:
            # Random gain
            if np.random.random() < 0.5:
                audio = audio * np.random.uniform(0.8, 1.2)
            # Time shift
            if np.random.random() < 0.5:
                shift = np.random.randint(-CONFIG['sr'], CONFIG['sr'])
                audio = np.roll(audio, shift)
        
        audio = preprocess(audio)
        mel = audio_to_mel(audio, CONFIG['sr'])
        
        # SpecAugment
        if self.augment:
            mel = spec_augment(mel)
        
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_tensor, label

# Dataset for TEST mashups
class MashupDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            audio, _ = librosa.load(path, sr=CONFIG['sr'], duration=CONFIG['duration'])
            target_len = CONFIG['sr'] * CONFIG['duration']
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
        except:
            audio = np.zeros(CONFIG['sr'] * CONFIG['duration'])
        
        audio = preprocess(audio)
        mel = audio_to_mel(audio, CONFIG['sr'])
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_tensor

In [60]:
# Prepare data
np.random.seed(42)
random.shuffle(train_data)

# Subsample if needed
n_samples = min(CONFIG['train_samples'], len(train_data))
train_data = train_data[:n_samples]

# Split
val_size = int(0.15 * len(train_data))
val_data = train_data[:val_size]
train_data_split = train_data[val_size:]

print(f"Train: {len(train_data_split)}, Val: {len(val_data)}")

train_loader = DataLoader(
    StemDataset(train_data_split, augment=True),
    batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2
)
val_loader = DataLoader(
    StemDataset(val_data, augment=False),
    batch_size=CONFIG['batch_size'], num_workers=2
)

Train: 850, Val: 150


In [61]:
# Model - Auto-detect feature dimensions
class GenreClassifier(nn.Module):
    def __init__(self, model_name='efficientnet_b0'):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        # Auto-detect feature size
        num_features = self.backbone.num_features
        print(f"Backbone features: {num_features}")
        
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CONFIG['num_classes'])
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

# Model options (uncomment one):
MODEL_NAME = 'efficientnet_b0'      # Fast, good accuracy
# MODEL_NAME = 'efficientnet_b2'    # Slower, better accuracy
# MODEL_NAME = 'resnet34'           # Classic, reliable

model = GenreClassifier(MODEL_NAME).to(device)
print(f"Model: {MODEL_NAME}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

Backbone features: 1280
Model: efficientnet_b0
Total params: 4,338,054


In [62]:
# Mixup
def mixup_data(x, y, alpha=0.3):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [63]:
# Training
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['lr'], epochs=CONFIG['epochs'], steps_per_epoch=len(train_loader)
)

best_acc = 0
for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        
        # Mixup
        data, target_a, target_b, lam = mixup_data(data, target, CONFIG['mixup_alpha'])
        
        optimizer.zero_grad()
        output = model(data)
        loss = mixup_criterion(criterion, output, target_a, target_b, lam)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        train_correct += (lam * (output.argmax(1) == target_a).float() + 
                         (1-lam) * (output.argmax(1) == target_b).float()).sum().item()
        train_total += target.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_correct += (output.argmax(1) == target).sum().item()
            val_total += target.size(0)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}, Best={best_acc:.4f}")

print(f"\nBest Val Accuracy: {best_acc:.4f}")

Epoch 1: 100%|██████████| 36/36 [01:28<00:00,  2.45s/it, loss=2.0445]


Epoch 1: Train=0.2371, Val=0.4533, Best=0.4533


Epoch 2: 100%|██████████| 36/36 [01:27<00:00,  2.43s/it, loss=1.4500]


Epoch 2: Train=0.5110, Val=0.6933, Best=0.6933


Epoch 3: 100%|██████████| 36/36 [01:28<00:00,  2.47s/it, loss=2.0509]


Epoch 3: Train=0.5691, Val=0.7533, Best=0.7533


Epoch 4: 100%|██████████| 36/36 [01:26<00:00,  2.41s/it, loss=1.3987]


Epoch 4: Train=0.6727, Val=0.8067, Best=0.8067


Epoch 5: 100%|██████████| 36/36 [01:28<00:00,  2.47s/it, loss=1.5758]


Epoch 5: Train=0.7071, Val=0.8467, Best=0.8467

Best Val Accuracy: 0.8467


In [68]:
# Test inference - CAREFUL about ID alignment
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# First, look at the sample submission to understand expected format
print("="*50)
print("CHECKING SUBMISSION FORMAT")
print("="*50)

if os.path.exists(SAMPLE_SUB):
    sample_sub = pd.read_csv(SAMPLE_SUB)
    print(f"\nSample submission:")
    print(f"  Columns: {sample_sub.columns.tolist()}")
    print(f"  Shape: {sample_sub.shape}")
    print(f"  First 5 rows:\n{sample_sub.head()}")
    print(f"  ID examples: {sample_sub.iloc[:5, 0].tolist()}")

if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    print(f"\nTest CSV:")
    print(f"  Columns: {test_df.columns.tolist()}")
    print(f"  Shape: {test_df.shape}")
    print(f"  First 5 rows:\n{test_df.head()}")

# Check what files actually exist in mashups folder
print(f"\nMashups folder contents (first 5):")
mashup_files = sorted(os.listdir(TEST_DIR))[:5]
print(f"  {mashup_files}")

# Build test files list IN THE ORDER OF SAMPLE SUBMISSION
# This is critical for correct alignment!
id_col = sample_sub.columns[0]  # First column is usually ID
print(f"\nUsing ID column: '{id_col}'")

test_files = []
test_ids = []
for idx, row in sample_sub.iterrows():
    test_id = row[id_col]
    # Try different filename patterns
    possible_names = [
        f"song{test_id}.wav",
        f"{test_id}",
        test_id if str(test_id).endswith('.wav') else f"{test_id}.wav"
    ]
    
    found = False
    for name in possible_names:
        path = os.path.join(TEST_DIR, name)
        if os.path.exists(path):
            test_files.append(path)
            test_ids.append(test_id)
            found = True
            break
    
    if not found:
        print(f"WARNING: File not found for ID: {test_id}")
        test_files.append(None)
        test_ids.append(test_id)

print(f"\nTest files found: {len([f for f in test_files if f is not None])} / {len(sample_sub)}")
print(f"Sample paths: {test_files[:3]}")

CHECKING SUBMISSION FORMAT

Sample submission:
  Columns: ['id', 'genre']
  Shape: (3020, 2)
  First 5 rows:
   id      genre
0   1       jazz
1   2      blues
2   3  classical
3   4        pop
4   5      disco
  ID examples: [1, 2, 3, 4, 5]

Test CSV:
  Columns: ['id', 'filename']
  Shape: (3020, 2)
  First 5 rows:
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav

Mashups folder contents (first 5):
  ['song0001.wav', 'song0002.wav', 'song0003.wav', 'song0004.wav', 'song0005.wav']

Using ID column: 'id'

Test files found: 2021 / 3020
Sample paths: [None, None, None]


In [65]:
# Run inference - handle missing files
class MashupDatasetSafe(Dataset):
    def __init__(self, file_paths):
        # Filter out None paths
        self.file_paths = [f for f in file_paths if f is not None]
        self.valid_indices = [i for i, f in enumerate(file_paths) if f is not None]
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            audio, _ = librosa.load(path, sr=CONFIG['sr'], duration=CONFIG['duration'])
            target_len = CONFIG['sr'] * CONFIG['duration']
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
        except Exception as e:
            print(f"Error loading {path}: {e}")
            audio = np.zeros(CONFIG['sr'] * CONFIG['duration'])
        
        audio = preprocess(audio)
        mel = audio_to_mel(audio, CONFIG['sr'])
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_tensor, self.valid_indices[idx]  # Return index for alignment

test_dataset = MashupDatasetSafe(test_files)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], num_workers=2, shuffle=False)

print(f"Running inference on {len(test_dataset)} files...")

# Store predictions with their indices
all_predictions = {}
with torch.no_grad():
    for data, indices in tqdm(test_loader, desc="Testing"):
        data = data.to(device)
        output = model(data)
        preds = output.argmax(1).cpu().numpy()
        for pred, idx in zip(preds, indices.numpy()):
            all_predictions[idx] = pred

print(f"Predictions made: {len(all_predictions)}")

Running inference on 0 files...


Testing: 0it [00:00, ?it/s]

Predictions made: 0


In [66]:
# Create submission with CORRECT ID alignment
submission = sample_sub.copy()

# Get the genre column name (second column)
genre_col = sample_sub.columns[1]
print(f"Genre column: '{genre_col}'")

# Fill predictions in order
predicted_genres = []
for idx in range(len(sample_sub)):
    if idx in all_predictions:
        pred_idx = all_predictions[idx]
        predicted_genres.append(GENRES[pred_idx])
    else:
        # Default to most common genre if file missing
        predicted_genres.append('rock')
        print(f"WARNING: No prediction for index {idx}, using default")

submission[genre_col] = predicted_genres

# Save
submission.to_csv('submission.csv', index=False)
print("\n" + "="*50)
print("SUBMISSION CREATED")
print("="*50)
print(f"\nShape: {submission.shape}")
print(f"Columns: {submission.columns.tolist()}")
print(f"\nFirst 10 rows:")
print(submission.head(10))
print(f"\nPrediction distribution:")
print(submission[genre_col].value_counts())

# Verify format matches sample
print(f"\n--- Verification ---")
print(f"Sample submission columns: {sample_sub.columns.tolist()}")
print(f"Our submission columns: {submission.columns.tolist()}")
print(f"Match: {list(sample_sub.columns) == list(submission.columns)}")

Genre column: 'genre'

SUBMISSION CREATED

Shape: (3020, 2)
Columns: ['id', 'genre']

First 10 rows:
   id genre
0   1  rock
1   2  rock
2   3  rock
3   4  rock
4   5  rock
5   6  rock
6   7  rock
7   8  rock
8   9  rock
9  10  rock

Prediction distribution:
genre
rock    3020
Name: count, dtype: int64

--- Verification ---
Sample submission columns: ['id', 'genre']
Our submission columns: ['id', 'genre']
Match: True


In [67]:
# Assessment
print("\n" + "="*50)
print("MODEL ASSESSMENT")
print("="*50)
print(f"Best Val Accuracy: {best_acc:.4f}")
print(f"\nKey settings:")
print(f"  - Duration: {CONFIG['duration']}s")
print(f"  - Noise augmentation: {CONFIG['noise_prob']*100:.0f}% prob")
print(f"  - Noise level: {CONFIG['noise_level']}")
print(f"  - Mixup: {CONFIG['mixup_alpha']}")
print(f"  - SpecAugment: Yes")

if best_acc >= 0.75:
    print("\nVERDICT: Strong! Scale up for better results.")
elif best_acc >= 0.60:
    print("\nVERDICT: Decent. Consider more noise augmentation.")
else:
    print("\nVERDICT: Needs work. Try different approach.")


MODEL ASSESSMENT
Best Val Accuracy: 0.8467

Key settings:
  - Duration: 10s
  - Noise augmentation: 70% prob
  - Noise level: (0.05, 0.3)
  - Mixup: 0.3
  - SpecAugment: Yes

VERDICT: Strong! Scale up for better results.


In [69]:
                                                                                                                            
# FIX: Correct file mapping                                                                                                                 
test_df = pd.read_csv(TEST_CSV)                                                                                                             
sample_sub = pd.read_csv(SAMPLE_SUB)                                                                                                        
                                                                                                                                          
id_to_filename = dict(zip(test_df['id'], test_df['filename']))                                                                              
                                                                                                                                          
test_files = []                                                                                                                             
for idx, row in sample_sub.iterrows():                                                                                                      
  filename = id_to_filename[row['id']]                                                                                                    
  filepath = os.path.join(BASE_PATH, filename)                                                                                            
  test_files.append(filepath)                                                                                                             
                                                                                                                                          
print(f"Files: {len(test_files)}")                                                                                                          
print(f"First 3: {test_files[:3]}")                                                                                                         
                                                                                                                                          
                                                                                                                   
# FIX: Run inference again                                                                                                                  
test_loader = DataLoader(MashupDataset(test_files), batch_size=CONFIG['batch_size'], num_workers=2)                                         
                                                                                                                                          
predictions = []                                                                                                                            
with torch.no_grad():                                                                                                                       
  for data in tqdm(test_loader, desc="Testing"):                                                                                          
      data = data.to(device)                                                                                                              
      output = model(data)                                                                                                                
      predictions.extend(output.argmax(1).cpu().numpy())                                                                                  
                                                                                                                                          
print(f"Predictions: {len(predictions)}")                                                                                                   
                                                                                                                                          
                                                                                                               
# FIX: Create correct submission                                                                                                            
submission = sample_sub.copy()                                                                                                              
submission['genre'] = [GENRES[p] for p in predictions]                                                                                      
submission.to_csv('submission.csv', index=False)                                                                                            
print(submission.head(10))                                                                                                                  
print(submission['genre'].value_counts())         

Files: 3020
First 3: ['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/song0001.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/song0002.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/song0003.wav']


Testing: 100%|██████████| 126/126 [01:49<00:00,  1.15it/s]

Predictions: 3020
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7      metal
7   8        pop
8   9        pop
9  10      disco
genre
pop          586
metal        470
hiphop       423
disco        296
reggae       294
blues        273
classical    194
jazz         193
rock         190
country      101
Name: count, dtype: int64
